In [4]:
import pandas as pd
from openai import OpenAI
from tqdm import tqdm
import time
import json

# Initialize client
client = OpenAI()

# Load human-coded data (300 cases only)
df = pd.read_csv("sample_for_llm_full.csv")

# Category whitelist
VALID_LABELS = {
    "Economy",
    "Healthcare",
    "Politics/Democracy",
    "Social Issues",
    "Crime",
    "Immigration",
    "Environment",
    "Foreign Policy",
    "Other",
    "Unclear"
}

# Prompt template
PROMPT_TEMPLATE = """
You are a survey research assistant coding open-ended responses to a public opinion question:

“Mention one most important problem facing the country.”

Your task is to assign EACH response to exactly ONE of the following categories:

- Economy
- Healthcare
- Politics/Democracy
- Social Issues
- Crime
- Immigration
- Environment
- Foreign Policy
- Other
- Unclear

DEFINITION AND DECISION RULES:

1. Assign only ONE category per response.
2. If multiple issues are mentioned, code the most emphasized issue. If equal emphasis, code the first issue.
3. If a response criticizes government, leaders, institutions, elections, democracy, corruption, polarization, or partisanship without naming a specific policy area, code as Politics/Democracy.
4. Responses referencing social division, polarization, cultural conflict, racial tension, or lack of social cohesion without explicit reference to political institutions should be coded as Social Issues.
5. Economic issues include inflation, prices, wages, jobs, debt, cost of living, taxes, and economic inequality.
6. Healthcare includes access, affordability, insurance, hospitals, abortion, and public health.
7. Immigration includes borders, illegal immigration, asylum, migrants, or refugee policy.
8. Crime includes violence, drugs, law enforcement, public safety, and incarceration.
9. Environment includes climate change, pollution, environmental protection, and natural disasters.
10. Foreign Policy includes wars, international relations, foreign governments, and national security abroad.
11. Responses that are vague, uninterpretable, or say “don’t know” should be coded as Unclear.
12. Responses that do not clearly fit any category should be coded as Other.

OUTPUT RULES:
Return your answer in valid JSON ONLY, using the following format:

{{
  "llm_code_label": "<ONE category label from the list exactly as written>",
  "confidence": "<High | Medium | Low>",
  "raw_model_output": "<brief explanation of why this category was chosen>"
}}

Rules:
- Choose exactly ONE category label.
- The category label must exactly match the list provided.
- Do not include any text outside the JSON object.


Response to code:
"{response_text}"
""".strip()

def classify_response(text):
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        temperature=0,
        messages=[
            {"role": "user", "content": PROMPT_TEMPLATE.format(response_text=text)}
        ],
    )

    raw_text = response.choices[0].message.content.strip()

    try:
        parsed = json.loads(raw_text)

        llm_code_label = parsed.get("llm_code_label", "Unclear")
        confidence = parsed.get("confidence", "Low")
        raw_model_output = parsed.get("raw_model_output", "")

        if llm_code_label not in VALID_LABELS:
            llm_code_label = "Invalid"

    except Exception as e:
        llm_code_label = "Unclear"
        confidence = "Low"
        raw_model_output = f"Parsing error. Raw output: {raw_text}"

    return llm_code_label, confidence, raw_model_output

llm_code_labels = []
confidence_scores = []
raw_outputs = []

for text in tqdm(df["response_text"]):
    label, confidence, explanation = classify_response(text)

    llm_code_labels.append(label)
    confidence_scores.append(confidence)
    raw_outputs.append(explanation)

    time.sleep(0.2)

# Attach results
df["llm_code_label"] = llm_code_labels
df["confidence"] = confidence_scores
df["raw_model_output"] = raw_outputs

# Save output
df.to_csv("llm_coded_sample_full.csv", index=False)

print("LLM coding complete. Results saved to llm_coded_sample.csv")


100%|██████████| 4960/4960 [2:11:54<00:00,  1.60s/it]   

LLM coding complete. Results saved to llm_coded_sample.csv
